
# QCVV Bridge: Rydberg CZ Gate Noise → RB Decay

This notebook connects **open-system noise modeling** for neutral-atom CZ gates to
**randomized benchmarking (RB)-style decay** and gate characterization.

## Goals
- Simulate fidelity decay vs gate depth using a simple effective noise model
- Fit a standard RB curve: $A p^m + B$
- Show where simple RB fits begin to miss structure
- Compare RB-style decay against an effective noise coordinate,
  $\gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi}$

This notebook is designed as a **QCVV bridge**: not a full hardware pipeline, but a compact,
interpretable demonstration of how open-system modeling can inform characterization workflows.



## Imports
We use NumPy/SciPy for simulation and fitting, and Matplotlib for plotting.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(7)



## Model setup

We define a compact effective noise coordinate
\[
\gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi},
\]
where:
- $\gamma$ is a relaxation / incoherent-noise scale
- $\gamma_{\phi}$ is a dephasing scale
- $\lambda$ weights dephasing contributions in the effective description

To keep the notebook lightweight and interpretable, we use a phenomenological decay model:
- a dominant exponential envelope controlled by $\gamma_{\mathrm{eff}}$
- a small coherent / structured correction term to mimic deviations from ideal RB behavior


In [ ]:

# Effective noise parameters
gamma = 0.018          # incoherent noise scale
gamma_phi = 0.010      # dephasing scale
lam = 0.75             # dephasing weight
gamma_eff = gamma + lam * gamma_phi

# Gate / control placeholders (not fully simulated here, but included for context)
Omega = 1.00
Delta = 0.35
V = 1.20

# Gate depths
m = np.arange(0, 61)

gamma_eff



## Simulate fidelity decay

We use a simple structured decay model:
\[
F(m) \approx A \exp(-\gamma_{\mathrm{eff}} m) + B
\]
plus a small oscillatory correction and weak quadratic drift to mimic
non-Markovian / coherent structure that standard RB may not fully capture.


In [ ]:

# "Ground-truth" synthetic decay parameters
A_true = 0.48
B_true = 0.50

# Structured synthetic fidelity data
coherent_amp = 0.010
coherent_freq = 0.22
drift_amp = 0.00006

fidelity_true = (
    A_true * np.exp(-gamma_eff * m) + B_true
    + coherent_amp * np.cos(coherent_freq * m) * np.exp(-0.015 * m)
    - drift_amp * (m**2)
)

# Add small measurement-like noise
noise_sigma = 0.003
fidelity_obs = fidelity_true + np.random.normal(0.0, noise_sigma, size=len(m))

# Clip for physical range
fidelity_obs = np.clip(fidelity_obs, 0.0, 1.0)

fidelity_obs[:5]



## Plot the synthetic fidelity data


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(m, fidelity_true, label="Synthetic underlying decay")
plt.scatter(m, fidelity_obs, s=18, label="Observed data")
plt.xlabel("Gate depth $m$")
plt.ylabel("Fidelity")
plt.title("Rydberg CZ gate: synthetic fidelity decay")
plt.legend()
plt.tight_layout()
plt.show()



## Fit a standard RB model

A common RB-inspired fit is
\[
F_{\mathrm{RB}}(m) = A p^m + B,
\]
where:
- $A$ is the contrast
- $p$ is the decay parameter
- $B$ is the asymptotic offset

If the noise is close to simple Markovian decay, this can fit very well.
If coherent structure or drift matters, residuals can reveal the mismatch.


In [ ]:

def rb_model(m, A, p, B):
    return A * (p ** m) + B

p0 = (0.45, 0.97, 0.50)
bounds = ([0.0, 0.0, 0.0], [1.0, 1.0, 1.0])

params_rb, cov_rb = curve_fit(rb_model, m, fidelity_obs, p0=p0, bounds=bounds, maxfev=20000)
A_fit, p_fit, B_fit = params_rb

fidelity_rb = rb_model(m, A_fit, p_fit, B_fit)

A_fit, p_fit, B_fit


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.scatter(m, fidelity_obs, s=18, label="Observed data")
plt.plot(m, fidelity_rb, linewidth=2.2, label=f"RB fit: A p^m + B,  p={p_fit:.4f}")
plt.xlabel("Gate depth $m$")
plt.ylabel("Fidelity")
plt.title("Standard RB-style fit to synthetic Rydberg decay")
plt.legend()
plt.tight_layout()
plt.show()



## Residuals: where does RB miss structure?
Residuals highlight whether the simple RB form captures the data cleanly.


In [ ]:

residuals_rb = fidelity_obs - fidelity_rb
rmse_rb = np.sqrt(np.mean(residuals_rb**2))

plt.figure(figsize=(8, 4.2))
plt.axhline(0.0, linewidth=1.2)
plt.scatter(m, residuals_rb, s=18)
plt.xlabel("Gate depth $m$")
plt.ylabel("Residual")
plt.title(f"RB residuals (RMSE = {rmse_rb:.5f})")
plt.tight_layout()
plt.show()

rmse_rb



## Compare to an effective-noise fit

Since the synthetic data was generated from a model organized around
$\gamma_{\mathrm{eff}}$, we can fit a phenomenological effective-noise curve:
\[
F_{\mathrm{eff}}(m) = A \exp(-\gamma_{\mathrm{eff}} m) + B.
\]

We keep $\gamma_{\mathrm{eff}}$ explicit to emphasize the modeling bridge:
open-system intuition can sometimes organize characterization better than a
purely agnostic RB fit.


In [ ]:

def eff_model(m, A, gamma_eff_fit, B):
    return A * np.exp(-gamma_eff_fit * m) + B

p0_eff = (0.45, 0.03, 0.50)
bounds_eff = ([0.0, 0.0, 0.0], [1.0, 1.0, 1.0])

params_eff, cov_eff = curve_fit(eff_model, m, fidelity_obs, p0=p0_eff, bounds=bounds_eff, maxfev=20000)
A_eff_fit, gamma_eff_fit, B_eff_fit = params_eff

fidelity_eff = eff_model(m, A_eff_fit, gamma_eff_fit, B_eff_fit)
residuals_eff = fidelity_obs - fidelity_eff
rmse_eff = np.sqrt(np.mean(residuals_eff**2))

A_eff_fit, gamma_eff_fit, B_eff_fit, rmse_eff


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.scatter(m, fidelity_obs, s=18, label="Observed data")
plt.plot(m, fidelity_rb, linewidth=2.0, label=f"RB fit (p={p_fit:.4f})")
plt.plot(m, fidelity_eff, linewidth=2.0, label=rf"Effective-noise fit ($\gamma_{{eff}}$={gamma_eff_fit:.4f})")
plt.xlabel("Gate depth $m$")
plt.ylabel("Fidelity")
plt.title("RB fit vs effective-noise fit")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 4.4))
plt.axhline(0.0, linewidth=1.2)
plt.plot(m, residuals_rb, marker='o', linestyle='-', label=f"RB residuals (RMSE={rmse_rb:.5f})")
plt.plot(m, residuals_eff, marker='s', linestyle='-', label=f"Eff residuals (RMSE={rmse_eff:.5f})")
plt.xlabel("Gate depth $m$")
plt.ylabel("Residual")
plt.title("Residual comparison")
plt.legend()
plt.tight_layout()
plt.show()



## Sweep the noise model

Next we vary the dephasing contribution $\gamma_{\phi}$ and track how:
- the effective coordinate $\gamma_{\mathrm{eff}}$ changes
- the RB decay parameter $p$ shifts
- fit quality degrades as structured noise increases

This gives a compact view of how a physically motivated noise coordinate
can organize changes in QCVV-style observables.


In [ ]:

gamma_phi_values = np.linspace(0.002, 0.028, 10)

gamma_eff_list = []
p_fit_list = []
rmse_list = []

for gphi in gamma_phi_values:
    geff = gamma + lam * gphi
    # Slightly increase coherent structure with dephasing for illustration
    coh_amp = coherent_amp * (1.0 + 8.0 * gphi)

    f = (
        A_true * np.exp(-geff * m) + B_true
        + coh_amp * np.cos(coherent_freq * m) * np.exp(-0.015 * m)
        - drift_amp * (m**2)
    )
    f = f + np.random.normal(0.0, noise_sigma, size=len(m))
    f = np.clip(f, 0.0, 1.0)

    try:
        pars, _ = curve_fit(rb_model, m, f, p0=p0, bounds=bounds, maxfev=20000)
        A_tmp, p_tmp, B_tmp = pars
        f_rb = rb_model(m, A_tmp, p_tmp, B_tmp)
        rmse_tmp = np.sqrt(np.mean((f - f_rb)**2))
    except RuntimeError:
        p_tmp = np.nan
        rmse_tmp = np.nan

    gamma_eff_list.append(geff)
    p_fit_list.append(p_tmp)
    rmse_list.append(rmse_tmp)

gamma_eff_arr = np.array(gamma_eff_list)
p_fit_arr = np.array(p_fit_list)
rmse_arr = np.array(rmse_list)


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(gamma_eff_arr, p_fit_arr, marker='o')
plt.xlabel(r"Effective noise $\gamma_{\mathrm{eff}}$")
plt.ylabel(r"RB decay parameter $p$")
plt.title(r"How $p$ shifts with $\gamma_{\mathrm{eff}}$")
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 4.8))
plt.plot(gamma_eff_arr, rmse_arr, marker='o')
plt.xlabel(r"Effective noise $\gamma_{\mathrm{eff}}$")
plt.ylabel("RB fit RMSE")
plt.title("RB fit quality across the noise sweep")
plt.tight_layout()
plt.show()



## Interpretation

A simple RB fit summarizes the decay with a single parameter $p$, which is useful.
But when structured open-system effects matter, an effective coordinate like
$\gamma_{\mathrm{eff}}$ can provide an additional organizing principle.

This is the bridge:
- **QCVV** asks how gates behave under repeated application and measurement
- **Open-system modeling** suggests which noise coordinates may better explain the behavior



## Takeaway

Open-system noise in Rydberg CZ gates can produce **structured deviations** from simple RB decay.
A compact effective coordinate such as
\[
\gamma_{\mathrm{eff}} = \gamma + \lambda \gamma_{\phi}
\]
can help organize these deviations and motivate extensions to standard QCVV analysis.

In a fuller workflow, this notebook can be extended to:
- ingest hardware data
- compare RB, interleaved RB, or cycle benchmarking fits
- tie fitted decay back to physically interpretable noise coordinates
